# Pipeline thi đua tự động — CVKD / TPKD / GDKD

Notebook này mở rộng pipeline doanh thu CVKD ban đầu bằng cách đưa toàn bộ
logic của `Contest_TPKD_latest.sql` (quản lý cấp 1 - BDM) và
`Contest_GDKD_latest.sql` (quản lý cấp 2 - BDD) vào Python, nhưng **không cố định tháng**.

Chỉ cần đổi `START_DATE` / `END_DATE` (hoặc bật `AUTO_END_DATE`), mọi cột tháng,
cột KPI và điều kiện thi đua sẽ tự sinh lại.

## Kiến trúc

1. Kéo **một lần** dữ liệu nền từ PostgreSQL ở dạng long:
   `(Code sale, nhân sự, QLC1, QLC2, Channel, Chức danh, Năm, Tháng, amount)`.
2. Kéo thêm bảng nhân sự và dữ liệu VietCredit (dùng cho phần trừ của GDKD).
3. Toàn bộ phép cộng theo tháng, pivot, tính %KPI và xét điều kiện thi đua
   được làm trong pandas → không còn `CASE WHEN "Tháng thanh toán" = 4 ...`.

## Thay đổi so với 2 file SQL gốc

| Vấn đề trong SQL gốc | Xử lý ở đây |
|---|---|
| JOIN nhân sự bằng `ON tên = tên OR sđt = sđt` → nhân bản doanh thu khi bảng nhân sự có dòng trùng | Dedup nhân sự bằng `ROW_NUMBER()`, join **SĐT trước → fallback Họ tên** (giữ nguyên logic notebook cũ) |
| Tháng cố định `IN (4,5,6,7,8)` và `= 2026` | Tham số `START_DATE` / `END_DATE`, cột tháng sinh tự động |
| Điều kiện thi đua hard-code cho đúng 5 tháng (`/ 5.0`) | Chia theo số tháng thực tế trong khoảng |
| `bonus_vietcredit` join vào CTE `qd1` không tồn tại trong file, và không lấy cột nào từ `qd1` | Bỏ join thừa này (kết quả không đổi) |
| `qlc2_trung_qlc1` được tính nhưng **không được dùng** ở `total_qlc2` → QLC2 kiêm QLC1 bị cộng doanh thu tự bán 2 lần | Có cờ `TRU_TU_BAN_KHI_QLC2_LA_QLC1` (mặc định `False` = giữ nguyên như SQL gốc) |


In [ ]:
# 1. IMPORT + CẤU HÌNH

import os
from datetime import date

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

# =========================
# DATABASE
# =========================
DB_USER = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Sales Aff"

DB_PASSWORD = os.getenv("SALES_AFF_DB_PASSWORD")

if not DB_PASSWORD:
    from getpass import getpass
    DB_PASSWORD = getpass("Nhập mật khẩu PostgreSQL: ")

# =========================
# KHOẢNG THỜI GIAN
# =========================
START_DATE = "2026-04-01"
END_DATE = "2026-08-31"

AUTO_END_DATE = False

if AUTO_END_DATE:
    END_DATE = date.today().isoformat()

START_DATE = pd.to_datetime(START_DATE).date()
END_DATE = pd.to_datetime(END_DATE).date()

if START_DATE > END_DATE:
    raise ValueError("START_DATE không được lớn hơn END_DATE.")

# Danh sách tháng (Period) dùng xuyên suốt notebook
PERIODS = pd.period_range(
    start=pd.Period(START_DATE, freq="M"),
    end=pd.Period(END_DATE, freq="M"),
    freq="M"
)

N_MONTHS = len(PERIODS)

print(f"Khoảng dữ liệu: {START_DATE} -> {END_DATE}")
print(f"Số tháng: {N_MONTHS}")
print("Các tháng:", [str(p) for p in PERIODS])


In [ ]:
# 2. CẤU HÌNH NGHIỆP VỤ (rút từ Contest_TPKD / Contest_GDKD)

# ---- CVKD: nhân viên kinh doanh ----
CVKD_CHANNELS = ["Core Agency", "Neo", "CTV_TSA (TSA2)"]
CVKD_TITLES   = ["DSA", "AG", "TSA 2"]

# ---- TPKD: quản lý cấp 1 (BDM) ----
TPKD_CHANNELS = ["Core Agency", "Neo", "TSA"]              # channel của chính BDM
TPKD_TITLES   = ["BDM", "BDM_DV", "SM", "TSA"]             # chức danh của chính BDM

# Doanh thu tự bán của BDM (SQL gốc KHÔNG gồm BDM_DV ở phần tự bán)
TPKD_SELF_CHANNELS = ["Core Agency", "Neo", "TSA"]
TPKD_SELF_TITLES   = ["BDM", "SM", "TSA"]

TPKD_TARGET = 200_000_000        # target/tháng của BDM

# Sale đặc biệt: doanh thu tính MPA riêng, cộng thẳng vào KPI quản lý của QLC1
SPECIAL_SALE_CODE = "0905099986"

# ---- GDKD: quản lý cấp 2 (BDD) ----
GDKD_CHANNELS = ["Core Agency", "Neo", "TSA", "H.O"]       # channel của chính BDD
GDKD_TITLES   = ["BDD", "BDD_DV", "SD", "TSA Manager"]     # chức danh của chính BDD

# Nhân viên được tính lên KPI quản lý của QLC1 (bản GDKD có thêm H.O / Kế toán)
GDKD_STAFF_CHANNELS = ["Core Agency", "Neo", "CTV_TSA (TSA2)", "H.O"]
GDKD_STAFF_TITLES   = ["DSA", "AG", "TSA 2", "Kế toán"]

# Doanh thu tự bán của BDM khi tính KPI cho nhánh GDKD
GDKD_BDM_SELF_CHANNELS = ["Core Agency", "Neo", "TSA"]
GDKD_BDM_SELF_TITLES   = ["BDM", "SM", "TSA"]

# Doanh thu tự bán của chính BDD
GDKD_SELF_CHANNELS = ["Core Agency", "Neo", "TSA", "H.O"]
GDKD_SELF_TITLES   = ["BDD", "BDD_DV", "SD", "TSA Manager"]

GDKD_TARGET = 450_000_000        # target/tháng của BDD

# SQL gốc có CTE qlc2_trung_qlc1 nhưng KHÔNG dùng -> QLC2 kiêm QLC1
# bị cộng doanh thu tự bán 2 lần.
# False = giữ nguyên hành vi SQL gốc. True = sửa lỗi trùng.
TRU_TU_BAN_KHI_QLC2_LA_QLC1 = False

# ---- Điều kiện thi đua ----
NGUONG_KPI_THANG = 0.70          # 1 tháng được tính là "đạt" khi >= 70% target
SO_THANG_DAT_TOI_THIEU = 3       # điều kiện 1
NGUONG_KPI_TRUNG_BINH = 0.50     # điều kiện 2

# Nếu khoảng báo cáo ngắn hơn 5 tháng thì không thể yêu cầu đủ 3 tháng
SO_THANG_DAT_TOI_THIEU = min(SO_THANG_DAT_TOI_THIEU, N_MONTHS)

print("Đã nạp cấu hình nghiệp vụ.")
print(f"Target BDM: {TPKD_TARGET:,} / tháng")
print(f"Target BDD: {GDKD_TARGET:,} / tháng")
print(f"Điều kiện: >= {NGUONG_KPI_THANG:.0%} trong {SO_THANG_DAT_TOI_THIEU}/{N_MONTHS} tháng "
      f"VÀ trung bình >= {NGUONG_KPI_TRUNG_BINH:.0%}")


In [ ]:
# 3. KẾT NỐI POSTGRESQL

connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(connection_string, pool_pre_ping=True)

with engine.connect() as conn:
    if conn.execute(text("SELECT 1")).scalar() != 1:
        raise ConnectionError("Không kiểm tra được kết nối PostgreSQL.")

print("Kết nối PostgreSQL thành công.")


In [ ]:
# 4. SQL

# Dedup bảng nhân sự để JOIN không nhân bản doanh thu.
# Đây là điểm khác biệt quan trọng so với 2 file SQL gốc (dùng ON ... OR ...).
NS_CTE = """
WITH ns_phone AS (
    SELECT * FROM (
        SELECT
            d.*,
            ROW_NUMBER() OVER (
                PARTITION BY TRIM(LEADING '0' FROM TRIM(d."Điện thoại"))
                ORDER BY d."Họ tên"
            ) AS rn
        FROM public.ds_nhan_su_affina d
        WHERE d."Điện thoại" IS NOT NULL
          AND TRIM(d."Điện thoại") <> ''
    ) x WHERE rn = 1
),
ns_name AS (
    SELECT * FROM (
        SELECT
            d.*,
            ROW_NUMBER() OVER (
                PARTITION BY UPPER(TRIM(d."Họ tên"))
                ORDER BY d."Điện thoại"
            ) AS rn
        FROM public.ds_nhan_su_affina d
        WHERE d."Họ tên" IS NOT NULL
          AND TRIM(d."Họ tên") <> ''
    ) x WHERE rn = 1
)
"""

# --- 4.1 Dữ liệu nền: doanh thu long theo (nhân sự, tháng) ---
SQL_BASE = NS_CTE + """
SELECT
    TRIM(uadcd."Code sale")                                  AS "Code sale",
    COALESCE(p."Họ tên",               n."Họ tên")            AS "Họ tên",
    COALESCE(p."Điện thoại",           n."Điện thoại")        AS "Điện thoại",
    COALESCE(p."Code UM",              n."Code UM")           AS "Code UM",
    COALESCE(p."Column1",              n."Column1")           AS "Code QLC2",
    COALESCE(p."QUẢN LÝ CẤP 1 (BDM)",  n."QUẢN LÝ CẤP 1 (BDM)") AS "QUẢN LÝ CẤP 1 (BDM)",
    COALESCE(p."QUẢN LÝ CẤP 2 (BDD)",  n."QUẢN LÝ CẤP 2 (BDD)") AS "QUẢN LÝ CẤP 2 (BDD)",
    COALESCE(p."Quản lý Cấp 3 (BDH)",  n."Quản lý Cấp 3 (BDH)") AS "Quản lý Cấp 3 (BDH)",
    COALESCE(p."Channal",              n."Channal")           AS "Channal",
    COALESCE(p."Chức danh",            n."Chức danh")         AS "Chức danh",
    COALESCE(p."Phòng ban",            n."Phòng ban")         AS "Phòng ban",

    EXTRACT(YEAR  FROM uadcd."Ngày thanh toán"::date)::int    AS "Năm thanh toán",
    EXTRACT(MONTH FROM uadcd."Ngày thanh toán"::date)::int    AS "Tháng thanh toán",

    SUM(uadcd."Số tiền thanh toán")                           AS amount

FROM public.union_all_data_cap_don uadcd

-- Match SĐT trước
LEFT JOIN ns_phone p
    ON TRIM(LEADING '0' FROM TRIM(uadcd."Code sale"))
     = TRIM(LEADING '0' FROM TRIM(p."Điện thoại"))

-- Chỉ fallback sang Họ tên khi KHÔNG match được SĐT
LEFT JOIN ns_name n
    ON p."Điện thoại" IS NULL
   AND UPPER(TRIM(uadcd."Code sale")) = UPPER(TRIM(n."Họ tên"))

WHERE uadcd."Ngày thanh toán"::date >= CAST(:start_date AS DATE)
  AND uadcd."Ngày thanh toán"::date <  CAST(:end_date AS DATE) + INTERVAL '1 day'

GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13
"""

# --- 4.2 Bảng nhân sự (dùng để map chức danh/channel cho BDM, BDD) ---
SQL_NHAN_SU = """
SELECT
    "Họ tên",
    "Điện thoại",
    "Code UM",
    "Column1" AS "Code QLC2",
    "QUẢN LÝ CẤP 1 (BDM)",
    "QUẢN LÝ CẤP 2 (BDD)",
    "Quản lý Cấp 3 (BDH)",
    "Channal",
    "Chức danh",
    "Phòng ban"
FROM public.ds_nhan_su_affina
"""

# --- 4.3 VietCredit: phần bị TRỪ khỏi KPI của QLC2 ---
SQL_VIETCREDIT = NS_CTE + """
SELECT
    COALESCE(p."Họ tên", n."Họ tên", TRIM(uadcd."Code sale")) AS "Họ tên",
    TRIM(uadcd."Code sale")                                   AS "Code sale",
    EXTRACT(YEAR  FROM uadcd."Ngày thanh toán"::date)::int     AS "Năm thanh toán",
    EXTRACT(MONTH FROM uadcd."Ngày thanh toán"::date)::int     AS "Tháng thanh toán",
    SUM(uadcd."Số tiền thanh toán")                            AS amount

FROM public.union_all_data_cap_don uadcd

LEFT JOIN ns_phone p
    ON TRIM(LEADING '0' FROM TRIM(uadcd."Code sale"))
     = TRIM(LEADING '0' FROM TRIM(p."Điện thoại"))

LEFT JOIN ns_name n
    ON p."Điện thoại" IS NULL
   AND UPPER(TRIM(uadcd."Code sale")) = UPPER(TRIM(n."Họ tên"))

WHERE uadcd."Loại bảo hiểm" = 'BHOTO'
  AND uadcd."Note" ILIKE '%VietCredit%'
  AND uadcd."Ngày thanh toán"::date >= CAST(:start_date AS DATE)
  AND uadcd."Ngày thanh toán"::date <  CAST(:end_date AS DATE) + INTERVAL '1 day'

GROUP BY 1,2,3,4
"""

print("Đã nạp 3 câu SQL.")


In [ ]:
# 5. CHẠY SQL

params = {"start_date": START_DATE, "end_date": END_DATE}

with engine.connect() as conn:
    df_base = pd.read_sql(text(SQL_BASE), conn, params=params)
    df_ns = pd.read_sql(text(SQL_NHAN_SU), conn)
    df_vc = pd.read_sql(text(SQL_VIETCREDIT), conn, params=params)

for df in (df_base, df_vc):
    if not df.empty:
        df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0.0)

if df_base.empty:
    raise ValueError(f"Không có dữ liệu từ {START_DATE} đến {END_DATE}.")

print(f"df_base    : {len(df_base):,} dòng")
print(f"df_ns      : {len(df_ns):,} dòng")
print(f"df_vc      : {len(df_vc):,} dòng")

display(df_base.head(10))


In [ ]:
# 6. HÀM TIỆN ÍCH

def norm_name(s):
    """Chuẩn hóa họ tên thành key so khớp."""
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.upper()
         .str.replace(r"\s+", " ", regex=True)
    )


def norm_phone(s):
    """Chuẩn hóa SĐT: bỏ khoảng trắng và số 0 ở đầu."""
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.replace(r"\D", "", regex=True)
         .str.lstrip("0")
    )


def month_cols(prefix):
    """Sinh tên cột tháng theo khoảng báo cáo, ví dụ total_2026_04."""
    return [f"{prefix}_{p.year}_{p.month:02d}" for p in PERIODS]


def add_month_key(df):
    df = df.copy()
    df["month_key"] = pd.PeriodIndex(
        pd.to_datetime(
            df["Năm thanh toán"].astype(int).astype(str)
            + "-"
            + df["Tháng thanh toán"].astype(int).astype(str).str.zfill(2)
            + "-01"
        ),
        freq="M"
    )
    return df


def pivot_months(df, key_cols, prefix):
    """
    Pivot dữ liệu long -> wide theo tháng.
    Luôn trả về đủ mọi tháng trong khoảng, tháng không có doanh thu = 0.
    """
    cols = month_cols(prefix)

    if df.empty:
        return pd.DataFrame(columns=key_cols + cols)

    t = (
        df.pivot_table(
            index=key_cols,
            columns="month_key",
            values="amount",
            aggfunc="sum",
            fill_value=0.0
        )
        .reset_index()
    )

    t = t.rename(columns={p: f"{prefix}_{p.year}_{p.month:02d}" for p in PERIODS})

    for c in cols:
        if c not in t.columns:
            t[c] = 0.0

    t.columns.name = None
    return t[key_cols + cols]


def attach(base, part, left_key, right_key, prefix):
    """Ghép bảng tháng (part) vào danh sách chính (base), NaN -> 0."""
    cols = month_cols(prefix)

    if part.empty:
        out = base.copy()
        for c in cols:
            out[c] = 0.0
        return out

    out = base.merge(
        part.rename(columns={right_key: "_k"}),
        how="left",
        left_on=left_key,
        right_on="_k"
    ).drop(columns=["_k"], errors="ignore")

    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out


def sum_prefixes(df, prefixes, out_prefix="total", signs=None):
    """Cộng/trừ các nhóm cột tháng lại thành một nhóm cột mới."""
    if signs is None:
        signs = [1] * len(prefixes)

    for p in PERIODS:
        suffix = f"{p.year}_{p.month:02d}"
        total = 0
        for prefix, sign in zip(prefixes, signs):
            total = total + sign * df[f"{prefix}_{suffix}"]
        df[f"{out_prefix}_{suffix}"] = total

    return df


def filter_rows(df, channels, titles):
    return df[
        df["Channal"].isin(channels)
        & df["Chức danh"].isin(titles)
    ]


def danh_gia_thi_dua(df, target, total_prefix="total"):
    """Tính %KPI từng tháng, số tháng đạt, KPI trung bình và kết quả thi đua."""
    kpi_cols = []

    for p in PERIODS:
        suffix = f"{p.year}_{p.month:02d}"
        kpi_col = f"KPI_{suffix}"
        df[kpi_col] = df[f"{total_prefix}_{suffix}"] / float(target)
        kpi_cols.append(kpi_col)

    df[total_prefix] = df[month_cols(total_prefix)].sum(axis=1)

    df["So_thang_dat_nguong"] = (df[kpi_cols] >= NGUONG_KPI_THANG).sum(axis=1)
    df["KPI_trung_binh"] = df[kpi_cols].sum(axis=1) / N_MONTHS

    dk1 = df["So_thang_dat_nguong"] >= SO_THANG_DAT_TOI_THIEU
    dk2 = df["KPI_trung_binh"] >= NGUONG_KPI_TRUNG_BINH

    df["DK1_nguong_thang"] = np.where(dk1, "Đạt", "Không đạt")
    df["DK2_TB"] = np.where(dk2, "Đạt", "Không đạt")
    df["Ket_qua_thi_dua"] = np.where(dk1 & dk2, "Đạt điều kiện thi đua", "Không đạt")

    df["_rank"] = (dk1 & dk2).astype(int)

    df = df.sort_values(
        by=["_rank", "So_thang_dat_nguong", "KPI_trung_binh", "Họ tên"],
        ascending=[False, False, False, True]
    ).drop(columns=["_rank"]).reset_index(drop=True)

    return df, kpi_cols


print("Đã nạp hàm tiện ích.")


In [ ]:
# 7. CHUẨN HÓA KEY

df_base = add_month_key(df_base)

# An toàn: chỉ giữ các tháng nằm trong khoảng báo cáo
ngoai_khoang = int((~df_base["month_key"].isin(PERIODS)).sum())
if ngoai_khoang:
    print(f"Cảnh báo: bỏ {ngoai_khoang:,} dòng nằm ngoài khoảng báo cáo.")
    df_base = df_base[df_base["month_key"].isin(PERIODS)].copy()

df_base["key_nv"]   = norm_name(df_base["Họ tên"])           # nhân viên bán hàng
df_base["key_qlc1"] = norm_name(df_base["QUẢN LÝ CẤP 1 (BDM)"])
df_base["key_qlc2"] = norm_name(df_base["QUẢN LÝ CẤP 2 (BDD)"])
df_base["code_sale_norm"] = df_base["Code sale"].astype("string").fillna("").str.strip()

df_ns["key_ten"]   = norm_name(df_ns["Họ tên"])
df_ns["key_phone"] = norm_phone(df_ns["Điện thoại"])

if not df_vc.empty:
    df_vc = add_month_key(df_vc)
    df_vc = df_vc[df_vc["month_key"].isin(PERIODS)].copy()
    df_vc["key_nv"] = norm_name(df_vc["Họ tên"])

print("Đã chuẩn hóa key.")


## A. CVKD — nhân viên kinh doanh

Giữ nguyên logic của notebook gốc: lọc theo Channel/Chức danh CVKD rồi tổng hợp theo nhân viên và tháng.


In [ ]:
# 8. CVKD

df_cvkd_rows = filter_rows(df_base, CVKD_CHANNELS, CVKD_TITLES)

df_cvkd = pivot_months(
    df_cvkd_rows,
    key_cols=["key_nv", "Họ tên", "Điện thoại", "Channal", "Chức danh"],
    prefix="total"
)

df_cvkd["total"] = df_cvkd[month_cols("total")].sum(axis=1)

df_cvkd = (
    df_cvkd
    .sort_values(by=["total"], ascending=False)
    .reset_index(drop=True)
)

print(f"Số CVKD: {len(df_cvkd):,}")
display(df_cvkd.head(10))


## B. TPKD — quản lý cấp 1 (BDM)

Theo `Contest_TPKD_latest.sql`:

```
KPI BDM = doanh thu quản lý (toàn bộ CVKD dưới BDM, trừ sale đặc biệt)
        + doanh thu đặc biệt của sale 0905099986
        + doanh thu tự bán của chính BDM
```

Target 200 triệu/tháng.


In [ ]:
# 9. DANH SÁCH BDM (ds_qly_cap1 -> ds_qly_c1_map_chuc_danh -> ds_truongphong)

def build_danh_sach_quan_ly(key_col, code_col, channels, titles):
    """
    Dựng danh sách quản lý (BDM hoặc BDD):
      1. Lấy tên quản lý duy nhất xuất hiện trong dữ liệu đơn hàng.
      2. Map sang chức danh / channel từ bảng nhân sự (ưu tiên Họ tên, fallback SĐT).
      3. Lọc theo channel + chức danh hợp lệ.
    """
    ds = (
        df_base.loc[
            (df_base[key_col] != "")
            & (~df_base[key_col].isin(["0", "O"])),
            [key_col, code_col]
        ]
        .copy()
    )

    ds[code_col] = ds[code_col].astype("string").fillna("").str.strip()

    ds = (
        ds.sort_values([key_col, code_col])
          .drop_duplicates(subset=[key_col], keep="first")
          .rename(columns={key_col: "key_ql", code_col: "Code sale"})
    )

    ds["key_phone"] = norm_phone(ds["Code sale"])

    # Map theo Họ tên
    ns_by_name = df_ns.drop_duplicates(subset=["key_ten"], keep="first")
    ds = ds.merge(
        ns_by_name[["key_ten", "Chức danh", "Channal", "Điện thoại"]],
        how="left", left_on="key_ql", right_on="key_ten"
    ).drop(columns=["key_ten"])

    # Fallback theo SĐT khi không map được tên
    ns_by_phone = (
        df_ns[df_ns["key_phone"] != ""]
        .drop_duplicates(subset=["key_phone"], keep="first")
    )
    ds = ds.merge(
        ns_by_phone[["key_phone", "Chức danh", "Channal", "Điện thoại"]],
        how="left", on="key_phone", suffixes=("", "_p")
    )

    for c in ["Chức danh", "Channal", "Điện thoại"]:
        ds[c] = ds[c].fillna(ds[f"{c}_p"])
        ds = ds.drop(columns=[f"{c}_p"])

    ds = ds[ds["Channal"].isin(channels) & ds["Chức danh"].isin(titles)]

    ds["Họ tên"] = ds["key_ql"]

    return ds[["key_ql", "Họ tên", "Code sale", "Điện thoại", "Channal", "Chức danh"]].reset_index(drop=True)


ds_truongphong = build_danh_sach_quan_ly(
    key_col="key_qlc1",
    code_col="Code UM",
    channels=TPKD_CHANNELS,
    titles=TPKD_TITLES
)

print(f"Số BDM trong danh sách thi đua: {len(ds_truongphong):,}")
display(ds_truongphong.head(10))


In [ ]:
# 10. TÍNH KPI TPKD

# 10.1 Doanh thu quản lý: CVKD dưới BDM, loại sale đặc biệt
rows_quanly = df_cvkd_rows[
    df_cvkd_rows["code_sale_norm"] != SPECIAL_SALE_CODE
]

tong_cvkd_theo_qlc1 = pivot_months(
    rows_quanly[rows_quanly["key_qlc1"] != ""],
    key_cols=["key_qlc1"],
    prefix="quanly"
)

# 10.2 Doanh thu đặc biệt của sale 0905099986 -> cộng vào QLC1 của sale đó
rows_special = df_base[df_base["code_sale_norm"] == SPECIAL_SALE_CODE]

dthu_dac_biet = pivot_months(
    rows_special[rows_special["key_qlc1"] != ""],
    key_cols=["key_qlc1"],
    prefix="dacbiet"
)

# 10.3 Doanh thu tự bán của chính BDM
rows_tuban_bdm = filter_rows(df_base, TPKD_SELF_CHANNELS, TPKD_SELF_TITLES)

truongphong_ptkd = pivot_months(
    rows_tuban_bdm[rows_tuban_bdm["key_nv"] != ""],
    key_cols=["key_nv"],
    prefix="tuban"
)

# 10.4 Gộp
df_tpkd = ds_truongphong.copy()
df_tpkd = attach(df_tpkd, tong_cvkd_theo_qlc1, "key_ql", "key_qlc1", "quanly")
df_tpkd = attach(df_tpkd, dthu_dac_biet,       "key_ql", "key_qlc1", "dacbiet")
df_tpkd = attach(df_tpkd, truongphong_ptkd,    "key_ql", "key_nv",   "tuban")

# KPI quản lý = quản lý thông thường + doanh thu đặc biệt
df_tpkd = sum_prefixes(df_tpkd, ["quanly", "dacbiet"], out_prefix="KPI_quan_ly")

# Tổng = KPI quản lý + tự bán
df_tpkd = sum_prefixes(df_tpkd, ["KPI_quan_ly", "tuban"], out_prefix="total")

df_tpkd, tpkd_kpi_cols = danh_gia_thi_dua(df_tpkd, TPKD_TARGET)

cols_tpkd = (
    ["Họ tên", "Code sale", "Channal", "Chức danh", "total"]
    + month_cols("KPI_quan_ly")
    + month_cols("tuban")
    + month_cols("total")
    + tpkd_kpi_cols
    + ["So_thang_dat_nguong", "KPI_trung_binh",
       "DK1_nguong_thang", "DK2_TB", "Ket_qua_thi_dua"]
)

df_tpkd = df_tpkd[cols_tpkd]

print(f"Số BDM: {len(df_tpkd):,}")
print(df_tpkd["Ket_qua_thi_dua"].value_counts().to_string())
display(df_tpkd.head(10))


## C. GDKD — quản lý cấp 2 (BDD)

Theo `Contest_GDKD_latest.sql`:

```
KPI QLC1 = doanh thu nhân viên dưới QLC1 + doanh thu tự bán của QLC1
KPI QLC2 = tổng KPI của các QLC1 thuộc QLC2
         + doanh thu tự bán của chính QLC2
         - doanh thu VietCredit (trùng tên)
```

Target 450 triệu/tháng.


In [ ]:
# 11. DANH SÁCH BDD

ds_giamdoc = build_danh_sach_quan_ly(
    key_col="key_qlc2",
    code_col="Code QLC2",
    channels=GDKD_CHANNELS,
    titles=GDKD_TITLES
)

print(f"Số BDD trong danh sách thi đua: {len(ds_giamdoc):,}")
display(ds_giamdoc.head(10))


In [ ]:
# 12. KPI CỦA TỪNG QLC1 (bản GDKD)

# 12.1 Doanh thu nhân viên dưới QLC1 (bản GDKD có thêm H.O / Kế toán)
rows_staff_gdkd = filter_rows(df_base, GDKD_STAFF_CHANNELS, GDKD_STAFF_TITLES)

dthu_qlc1 = pivot_months(
    rows_staff_gdkd[rows_staff_gdkd["key_qlc1"] != ""],
    key_cols=["key_qlc1"],
    prefix="quanly"
)

# 12.2 Doanh thu tự bán của chính QLC1
rows_tuban_qlc1 = filter_rows(df_base, GDKD_BDM_SELF_CHANNELS, GDKD_BDM_SELF_TITLES)

tu_ban_qlc1 = pivot_months(
    rows_tuban_qlc1[rows_tuban_qlc1["key_nv"] != ""],
    key_cols=["key_nv"],
    prefix="tuban"
)

# 12.3 KPI QLC1 = quản lý + tự bán
kpi_qlc1 = dthu_qlc1.copy()
kpi_qlc1 = attach(kpi_qlc1, tu_ban_qlc1, "key_qlc1", "key_nv", "tuban")
kpi_qlc1 = sum_prefixes(kpi_qlc1, ["quanly", "tuban"], out_prefix="kpi_qlc1")
kpi_qlc1 = kpi_qlc1[["key_qlc1"] + month_cols("kpi_qlc1")]

# 12.4 Mapping QLC1 -> QLC2 rồi cộng KPI QLC1 lên QLC2
mapping = (
    df_base.loc[
        (df_base["key_qlc1"] != "") & (df_base["key_qlc2"] != ""),
        ["key_qlc1", "key_qlc2"]
    ]
    .drop_duplicates()
)

kpi_qlc1_theo_qlc2 = (
    mapping.merge(kpi_qlc1, how="left", on="key_qlc1")
           .fillna({c: 0.0 for c in month_cols("kpi_qlc1")})
           .groupby("key_qlc2", as_index=False)[month_cols("kpi_qlc1")]
           .sum()
)

print(f"Số QLC1 có KPI  : {len(kpi_qlc1):,}")
print(f"Số cặp QLC1-QLC2: {len(mapping):,}")
display(kpi_qlc1_theo_qlc2.head(10))


In [ ]:
# 13. TÍNH KPI GDKD

# 13.1 Doanh thu tự bán của chính QLC2
rows_tuban_qlc2 = filter_rows(df_base, GDKD_SELF_CHANNELS, GDKD_SELF_TITLES)

giamdoc_ptkd = pivot_months(
    rows_tuban_qlc2[rows_tuban_qlc2["key_nv"] != ""],
    key_cols=["key_nv"],
    prefix="tuban"
)

# 13.2 VietCredit (phần bị trừ)
vietcredit = pivot_months(df_vc, key_cols=["key_nv"], prefix="vietcredit")

# 13.3 Gộp
df_gdkd = ds_giamdoc.copy()
df_gdkd = attach(df_gdkd, kpi_qlc1_theo_qlc2, "key_ql", "key_qlc2", "kpi_qlc1")
df_gdkd = attach(df_gdkd, giamdoc_ptkd,       "key_ql", "key_nv",   "tuban")
df_gdkd = attach(df_gdkd, vietcredit,         "key_ql", "key_nv",   "vietcredit")

# QLC2 đồng thời là QLC1 -> doanh thu tự bán đã nằm trong kpi_qlc1
qlc2_trung_qlc1 = set(
    mapping.loc[mapping["key_qlc1"] == mapping["key_qlc2"], "key_qlc2"]
)

df_gdkd["La_QLC1_kiem_QLC2"] = df_gdkd["key_ql"].isin(qlc2_trung_qlc1)

if TRU_TU_BAN_KHI_QLC2_LA_QLC1:
    mask = df_gdkd["La_QLC1_kiem_QLC2"]
    for c in month_cols("tuban"):
        df_gdkd.loc[mask, c] = 0.0
    print(f"Đã bỏ tự bán trùng cho {mask.sum():,} QLC2 kiêm QLC1.")

# total = KPI các QLC1 + tự bán QLC2 - VietCredit
df_gdkd = sum_prefixes(
    df_gdkd,
    ["kpi_qlc1", "tuban", "vietcredit"],
    out_prefix="total",
    signs=[1, 1, -1]
)

df_gdkd, gdkd_kpi_cols = danh_gia_thi_dua(df_gdkd, GDKD_TARGET)

cols_gdkd = (
    ["Họ tên", "Code sale", "Channal", "Chức danh", "La_QLC1_kiem_QLC2", "total"]
    + month_cols("kpi_qlc1")
    + month_cols("tuban")
    + month_cols("vietcredit")
    + month_cols("total")
    + gdkd_kpi_cols
    + ["So_thang_dat_nguong", "KPI_trung_binh",
       "DK1_nguong_thang", "DK2_TB", "Ket_qua_thi_dua"]
)

df_gdkd = df_gdkd[cols_gdkd]

print(f"Số BDD: {len(df_gdkd):,}")
print(df_gdkd["Ket_qua_thi_dua"].value_counts().to_string())
display(df_gdkd.head(10))


In [ ]:
# 14. KIỂM TRA

print("========== KIỂM TRA ==========")
print(f"Khoảng báo cáo      : {START_DATE} -> {END_DATE} ({N_MONTHS} tháng)")
print(f"Số dòng dữ liệu nền : {len(df_base):,}")
print(f"Tổng doanh thu nền  : {df_base['amount'].sum():,.0f}")
print()
print(f"CVKD  : {len(df_cvkd):,} người | doanh thu {df_cvkd['total'].sum():,.0f}")
print(f"TPKD  : {len(df_tpkd):,} người | tổng KPI {df_tpkd['total'].sum():,.0f}")
print(f"GDKD  : {len(df_gdkd):,} người | tổng KPI {df_gdkd['total'].sum():,.0f}")
print()

# CVKD: pivot phải khớp với dữ liệu nguồn
raw_cvkd = df_cvkd_rows["amount"].sum()
piv_cvkd = df_cvkd["total"].sum()

print(f"CVKD raw   : {raw_cvkd:,.0f}")
print(f"CVKD pivot : {piv_cvkd:,.0f}")

if abs(raw_cvkd - piv_cvkd) > 0.01:
    raise ValueError("Tổng doanh thu CVKD raw và pivot không khớp.")

print("✓ Tổng doanh thu CVKD khớp.")
print()

# Cảnh báo quản lý không map được chức danh
qlc1_all = set(df_base.loc[df_base["key_qlc1"] != "", "key_qlc1"])
qlc2_all = set(df_base.loc[df_base["key_qlc2"] != "", "key_qlc2"])

missing_bdm = qlc1_all - set(ds_truongphong["key_ql"])
missing_bdd = qlc2_all - set(ds_giamdoc["key_ql"])

print(f"QLC1 xuất hiện trong data nhưng KHÔNG vào danh sách BDM: {len(missing_bdm):,}")
print(f"QLC2 xuất hiện trong data nhưng KHÔNG vào danh sách BDD: {len(missing_bdd):,}")

if missing_bdm:
    print("  Ví dụ BDM:", list(sorted(missing_bdm))[:5])
if missing_bdd:
    print("  Ví dụ BDD:", list(sorted(missing_bdd))[:5])

# Đơn hàng không map được nhân sự
khong_map = df_base[df_base["key_nv"] == ""]
print()
print(f"Doanh thu KHÔNG map được nhân sự: {khong_map['amount'].sum():,.0f} "
      f"({len(khong_map):,} dòng)")


In [ ]:
# 15. XUẤT EXCEL

output_file = (
    f"thi_dua_{START_DATE.strftime('%Y%m%d')}_{END_DATE.strftime('%Y%m%d')}.xlsx"
)

drop_tech = ["month_key", "key_nv", "key_qlc1", "key_qlc2", "key_ql",
             "code_sale_norm", "key_phone"]

df_raw_export = df_base.drop(columns=drop_tech, errors="ignore")
df_cvkd_export = df_cvkd.drop(columns=drop_tech, errors="ignore")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_gdkd.to_excel(writer, sheet_name="GDKD_QLC2", index=False)
    df_tpkd.to_excel(writer, sheet_name="TPKD_QLC1", index=False)
    df_cvkd_export.to_excel(writer, sheet_name="CVKD", index=False)
    df_raw_export.to_excel(writer, sheet_name="Raw_Monthly", index=False)

    pd.DataFrame({
        "Tham số": [
            "START_DATE", "END_DATE", "Số tháng",
            "Target BDM/tháng", "Target BDD/tháng",
            "Ngưỡng KPI tháng", "Số tháng đạt tối thiểu", "Ngưỡng KPI trung bình",
            "Sale đặc biệt", "Trừ tự bán khi QLC2 kiêm QLC1"
        ],
        "Giá trị": [
            str(START_DATE), str(END_DATE), N_MONTHS,
            TPKD_TARGET, GDKD_TARGET,
            NGUONG_KPI_THANG, SO_THANG_DAT_TOI_THIEU, NGUONG_KPI_TRUNG_BINH,
            SPECIAL_SALE_CODE, TRU_TU_BAN_KHI_QLC2_LA_QLC1
        ]
    }).to_excel(writer, sheet_name="Tham_so", index=False)

print(f"Đã xuất Excel: {output_file}")
print(f"Đường dẫn: {os.path.abspath(output_file)}")


## Ghi chú vận hành

### Đổi kỳ báo cáo
Chỉ sửa `START_DATE` / `END_DATE` ở cell 1, hoặc đặt `AUTO_END_DATE = True`.
Mọi cột `total_YYYY_MM`, `KPI_YYYY_MM` và điều kiện thi đua tự sinh lại theo số tháng thực tế.

### Đổi tiêu chí thi đua
Sửa ở cell 2: `TPKD_TARGET`, `GDKD_TARGET`, `NGUONG_KPI_THANG`,
`SO_THANG_DAT_TOI_THIEU`, `NGUONG_KPI_TRUNG_BINH`, và các danh sách Channel/Chức danh.

### Những chỗ cần xác nhận lại với nghiệp vụ

1. **Chức danh tự bán của BDM.** SQL TPKD dùng `('BDM','SM','TSA')` cho phần tự bán
   nhưng `('BDM','BDM_DV','SM','TSA')` cho danh sách BDM. BDM_DV vì vậy không được
   tính doanh thu tự bán. Ở đây tách thành `TPKD_TITLES` và `TPKD_SELF_TITLES` để đổi được.

2. **QLC2 kiêm QLC1.** SQL GDKD có CTE `qlc2_trung_qlc1` nhưng không dùng, nên người
   vừa là QLC1 vừa là QLC2 bị cộng doanh thu tự bán hai lần. Cột `La_QLC1_kiem_QLC2`
   đánh dấu các trường hợp này; đặt `TRU_TU_BAN_KHI_QLC2_LA_QLC1 = True` để sửa.

3. **Mapping QLC1 → QLC2.** Nếu một QLC1 xuất hiện dưới nhiều QLC2 trong bảng nhân sự,
   KPI của QLC1 đó được cộng cho cả hai QLC2 (giống SQL gốc). Kiểm tra bảng `mapping`
   nếu con số tổng cao bất thường.

4. **Join nhân sự.** Notebook dùng SĐT trước → fallback Họ tên, và dedup bảng nhân sự
   bằng `ROW_NUMBER()`. Con số có thể thấp hơn SQL gốc (`ON ... OR ...`) vì SQL gốc
   nhân bản doanh thu khi có dòng nhân sự trùng. Cell 14 in ra phần doanh thu
   không map được nhân sự để đối chiếu.

5. **VietCredit.** SQL gốc join vào CTE `qd1` không tồn tại trong file và không dùng cột nào
   từ đó, nên phần join này đã được bỏ. Nếu `qd1` thực sự dùng để lọc theo hiệu lực sản phẩm,
   cần bổ sung lại vào `SQL_VIETCREDIT`.
